In [79]:
import pandas as pd
pd.set_option('display.float_format', lambda x: '%.6f' % x)

import numpy as np

def generate_expanded_sample():
    data = {
        'Start_employment': [
            # Group 1: Mixed (Pre/Post) - Primary & Client
            '032021', '012022', '012023', 
            # Group 2: Gaps & Forward Fill (Inflation)
            '062021', '012022', '022023',
            # Group 3: Pure Backfill (Deflation) - Only 2023 has data
            '012021', '012022', '012023'
        ],
        'End_employment': [
            '122021', '122022', '082023', 
            '122021', '122022', '112023',
            '122021', '122022', '122023'
        ],
        'Parent_employer_name': ['TechCorp']*3 + ['FinanceHub']*3 + ['RetailCorp']*3,
        'Subsidiary_employer_name': ['CloudDiv']*3 + ['BankNodes']*3 + ['StoreFront']*3,
        'Designation': ['Engineer']*3 + ['Analyst']*3 + ['Manager']*3,
        'Country': ['USA']*3 + ['UK']*3 + ['India']*3,
        
        # --- Primary Income Columns ---
        'Primary_income_pretax': [
            95000, np.nan, np.nan,      # Group 1
            np.nan, np.nan, np.nan,      # Group 2
            np.nan, np.nan, 1200000      # Group 3 (Backfill target)
        ],
        'Primary_income_posttax': [
            np.nan, 72000, 75000,      # Group 1
            np.nan, 52000, np.nan,      # Group 2
            np.nan, np.nan, np.nan       # Group 3
        ],
        'Primary_income_currency': [
            'USD', 'USD', 'USD', 
            np.nan, 'GBP', np.nan, 
            np.nan, np.nan, 'INR'
        ],

        # --- Client Declared Columns ---
        'Client_declared_pretax': [
            90000, np.nan, np.nan,      # Group 1
            45000, np.nan, 50000,       # Group 2
            np.nan, np.nan, np.nan       # Group 3
        ],
        'Client_declared_posttax': [
            np.nan, 68000, np.nan,      # Group 1
            np.nan, np.nan, np.nan,      # Group 2
            np.nan, np.nan, 900000       # Group 3 (Backfill target)
        ],
        'Client_declared_currency': [
            'USD', 'USD', np.nan, 
            'GBP', np.nan, 'GBP', 
            np.nan, np.nan, 'INR'
        ]
    }
    
    return pd.DataFrame(data)

# Initialize the new test bed
df = generate_expanded_sample()

In [90]:
# Helper: Tenure calculation (called once at the start)
def calculate_tenure_months(start_str, end_str):
    start_dt = datetime.strptime(start_str, "%m%Y")
    end_dt = datetime.strptime(end_str, "%m%Y")
    return (end_dt.year - start_dt.year) * 12 + (end_dt.month - start_dt.month) + 1

In [91]:
def identify_missing_rows(df, pre_col, post_col, flag_name):
    """
    Identifies the availability of income data for a specific pair of columns.
    Creates a flag column with the name provided in flag_name.
    """
    df_copy = df.copy()
    
    def get_flag(row):
        pre_exists = pd.notnull(row[pre_col])
        post_exists = pd.notnull(row[post_col])
        
        if pre_exists and post_exists:
            return "Both"
        elif pre_exists:
            return "Pre-tax"
        elif post_exists:
            return "Post-tax"
        else:
            return "Missing"

    df_copy[flag_name] = df_copy.apply(get_flag, axis=1)
    return df_copy

In [81]:
import pandas as pd
import numpy as np
from datetime import datetime

def get_inflation_factor(old_year, new_year, country):
    """Dummy inflation function: returns 1.05 per year difference."""
    diff = int(new_year) - int(old_year)
    return 1.05 ** diff

def calculate_tenure_months(start_str, end_str):
    """Calculates tenure in months from MMYYYY strings."""
    start_dt = datetime.strptime(start_str, "%m%Y")
    end_dt = datetime.strptime(end_str, "%m%Y")
    # Including the end month in the count
    return (end_dt.year - start_dt.year) * 12 + (end_dt.month - start_dt.month) + 1

In [92]:
def impute_income_group(df, pre_col, post_col, curr_col, meta_prefix):
    df_copy = df.copy()
    group_cols = ['Parent_employer_name', 'Subsidiary_employer_name', 'Designation', 'Country']
    
    # Meta columns for this specific set
    meta_cols = {
        'factor': f'{meta_prefix}_Inflation_Factor',
        'tenure': f'{meta_prefix}_Ref_Tenure',
        'period': f'{meta_prefix}_Ref_Time_Period',
        'ref_inc': f'{meta_prefix}_Ref_Income'
    }
    
    for col in meta_cols.values():
        df_copy[col] = np.nan

    def process_group(group):
        group = group.sort_values('Start_employment')
        indices = group.index.tolist()
        
        def is_missing(idx):
            return pd.isnull(group.loc[idx, pre_col]) and pd.isnull(group.loc[idx, post_col])

        # --- DEFLATED (Backward) ---
        for i in range(len(indices) - 2, -1, -1):
            curr_idx, next_idx = indices[i], indices[i+1]
            if is_missing(curr_idx):
                ref_pre = group.loc[next_idx, pre_col]
                ref_post = group.loc[next_idx, post_col]
                
                if pd.notnull(ref_pre) or pd.notnull(ref_post):
                    use_pre = pd.notnull(ref_pre)
                    ref_val = ref_pre if use_pre else ref_post
                    
                    ref_yr, curr_yr = group.loc[next_idx, 'End_employment'][-4:], group.loc[curr_idx, 'End_employment'][-4:]
                    factor = get_inflation_factor(curr_yr, ref_yr, group.loc[curr_idx, 'Country'])
                    
                    # USE PRE-CALCULATED TENURE
                    ref_t = group.loc[next_idx, 'row_tenure_months']
                    curr_t = group.loc[curr_idx, 'row_tenure_months']
                    
                    imputed_val = (ref_val / ref_t) / factor * curr_t
                    group.at[curr_idx, pre_col if use_pre else post_col] = imputed_val
                    group.at[curr_idx, curr_col] = group.loc[next_idx, curr_col]
                    
                    # Store Meta
                    group.at[curr_idx, meta_cols['factor']] = factor
                    group.at[curr_idx, meta_cols['tenure']] = ref_t
                    group.at[curr_idx, meta_cols['ref_inc']] = ref_val
                    group.at[curr_idx, meta_cols['period']] = f"{group.loc[next_idx, 'Start_employment']}-{group.loc[next_idx, 'End_employment']}"

        # --- INFLATED (Forward) ---
        for i in range(1, len(indices)):
            curr_idx, prev_idx = indices[i], indices[i-1]
            if is_missing(curr_idx):
                ref_pre = group.loc[prev_idx, pre_col]
                ref_post = group.loc[prev_idx, post_col]
                
                if pd.notnull(ref_pre) or pd.notnull(ref_post):
                    use_pre = pd.notnull(ref_pre)
                    ref_val = ref_pre if use_pre else ref_post
                    
                    ref_yr, curr_yr = group.loc[prev_idx, 'End_employment'][-4:], group.loc[curr_idx, 'End_employment'][-4:]
                    factor = get_inflation_factor(ref_yr, curr_yr, group.loc[curr_idx, 'Country'])
                    
                    # USE PRE-CALCULATED TENURE
                    ref_t = group.loc[prev_idx, 'row_tenure_months']
                    curr_t = group.loc[curr_idx, 'row_tenure_months']
                    
                    imputed_val = (ref_val / ref_t) * factor * curr_t
                    group.at[curr_idx, pre_col if use_pre else post_col] = imputed_val
                    group.at[curr_idx, curr_col] = group.loc[prev_idx, curr_col]
                    
                    group.at[curr_idx, meta_cols['factor']] = factor
                    group.at[curr_idx, meta_cols['tenure']] = ref_t
                    group.at[curr_idx, meta_cols['ref_inc']] = ref_val
                    group.at[curr_idx, meta_cols['period']] = f"{group.loc[prev_idx, 'Start_employment']}-{group.loc[prev_idx, 'End_employment']}"
        return group

    return df_copy.groupby(group_cols, group_keys=False).apply(process_group)

In [95]:
# --- 1. PRIMARY INCOME SET ---
df['row_tenure_months'] = df.apply(lambda x: calculate_tenure_months(x['Start_employment'], x['End_employment']), axis=1)
# Flag it
df_final = identify_missing_rows(
    df, 
    pre_col='Primary_income_pretax', 
    post_col='Primary_income_posttax', 
    flag_name='Primary_Missing_Flag'
)
# Impute it
df_final = impute_income_group(
    df_final, 
    pre_col='Primary_income_pretax', 
    post_col='Primary_income_posttax', 
    curr_col='Primary_income_currency',
    meta_prefix='Primary'
)

# --- 2. CLIENT DECLARED SET ---
# Flag it
df_final = identify_missing_rows(
    df_final, 
    pre_col='Client_declared_pretax', 
    post_col='Client_declared_posttax', 
    flag_name='Client_Missing_Flag'
)
# Impute it
df_final = impute_income_group(
    df_final, 
    pre_col='Client_declared_pretax', 
    post_col='Client_declared_posttax', 
    curr_col='Client_declared_currency',
    meta_prefix='Client'
)

C:\Users\User\AppData\Local\Temp\ipykernel_13828\4122162687.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '012022-122022' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  group.at[curr_idx, meta_cols['period']] = f"{group.loc[prev_idx, 'Start_employment']}-{group.loc[prev_idx, 'End_employment']}"
C:\Users\User\AppData\Local\Temp\ipykernel_13828\4122162687.py:49: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '012023-122023' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  group.at[curr_idx, meta_cols['period']] = f"{group.loc[next_idx, 'Start_employment']}-{group.loc[next_idx, 'End_employment']}"
C:\Users\User\AppData\Local\Temp\ipykernel_13828\4122162687.py:79: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns

In [96]:
def aggregate_employment_summary(df):
    df_copy = df.copy()
    group_cols = ['Parent_employer_name', 'Subsidiary_employer_name', 'Designation', 'Country']
    
    income_sets = {
        'Primary': ['Primary_income_pretax', 'Primary_income_posttax', 'Primary_income_currency'],
        'Client': ['Client_declared_pretax', 'Client_declared_posttax', 'Client_declared_currency']
    }

    def group_logic(group):
        res = {}
        for prefix, cols in income_sets.items():
            pre_col, post_col, curr_col = cols
            
            # Sums
            res[f'Total_{prefix}_Pretax'] = group[pre_col].sum()
            res[f'Total_{prefix}_Posttax'] = group[post_col].sum()
            res[f'Currency_{prefix}'] = group[curr_col].dropna().iloc[0] if not group[curr_col].dropna().empty else np.nan
            
            # Pre-tax Annualized (Normalized)
            pre_valid_rows = group[group[pre_col].notnull()]
            pre_tenure = pre_valid_rows['row_tenure_months'].sum()
            res[f'Annualized_Avg_{prefix}_Pretax'] = (res[f'Total_{prefix}_Pretax'] / pre_tenure * 12) if pre_tenure > 0 else np.nan
                
            # Post-tax Annualized (Normalized)
            post_valid_rows = group[group[post_col].notnull()]
            post_tenure = post_valid_rows['row_tenure_months'].sum()
            res[f'Annualized_Avg_{prefix}_Posttax'] = (res[f'Total_{prefix}_Posttax'] / post_tenure * 12) if post_tenure > 0 else np.nan
                
        return pd.Series(res)

    return df_copy.groupby(group_cols, observed=True).apply(group_logic).reset_index()

In [97]:
# 1. Infer for Primary Income
df_final = infer_pretax_income(
    df_final, 
    pre_col='Primary_income_pretax', 
    post_col='Primary_income_posttax', 
    meta_prefix='Primary'
)

# 2. Infer for Client Declared Income
df_final = infer_pretax_income(
    df_final, 
    pre_col='Client_declared_pretax', 
    post_col='Client_declared_posttax', 
    meta_prefix='Client'
)

In [98]:
df_final

,Start_employment,End_employment,Parent_employer_name,Subsidiary_employer_name,Designation,Country,Primary_income_pretax,Primary_income_posttax,Primary_income_currency,Client_declared_pretax,...,Client_Ref_Time_Period,Client_Ref_Income,Primary_Estimated_Tax_Paid,Primary_Tax_Rate,Primary_Tax_Source_Desc,Primary_Tax_Source_URL,Client_Estimated_Tax_Paid,Client_Tax_Rate,Client_Tax_Source_Desc,Client_Tax_Source_URL
4,012022,122022,FinanceHub,BankNodes,Analyst,UK,74285.710000,52000.000000,GBP,57142.857143,...,022023-112023,50000.000000,22285.710000,0.300000,Global Tax Engine v1.2,https://tax-engine.example.com/api/v1,NaN,NaN,NaN,NaN
5,022023,112023,FinanceHub,BankNodes,Analyst,UK,65000.000000,45500.000000,GBP,50000.000000,...,NaN,NaN,19500.000000,0.300000,Global Tax Engine v1.2,https://tax-engine.example.com/api/v1,NaN,NaN,NaN,NaN
3,062021,122021,FinanceHub,BankNodes,Analyst,UK,41269.840000,28888.888889,GBP,45000.000000,...,NaN,NaN,12380.950000,0.300000,Global Tax Engine v1.2,https://tax-engine.example.com/api/v1,NaN,NaN,NaN,NaN
6,012021,122021,RetailCorp,StoreFront,Manager,India,1088435.374150,NaN,INR,1166180.760000,...,012022-122022,857142.857143,NaN,NaN,NaN,NaN,349854.230000,0.300000,Global Tax Engine v1.2,https://tax-engine.example.com/api/v1
7,012022,122022,RetailCorp,StoreFront,Manager,India,1142857.142857,NaN,INR,1224489.800000,...,012023-122023,900000.000000,NaN,NaN,NaN,NaN,367346.940000,0.300000,Global Tax Engine v1.2,https://tax-engine.example.com/api/v1
8,012023,122023,RetailCorp,StoreFront,Manager,India,1200000.000000,NaN,INR,1285714.290000,...,NaN,NaN,NaN,NaN,NaN,NaN,385714.290000,0.300000,Global Tax Engine v1.2,https://tax-engine.example.com/api/v1
1,012022,122022,TechCorp,CloudDiv,Engineer,USA,102857.140000,72000.000000,USD,97142.860000,...,NaN,NaN,30857.140000,0.300000,Global Tax Engine v1.2,https://tax-engine.example.com/api/v1,29142.860000,0.300000,Global Tax Engine v1.2,https://tax-engine.example.com/api/v1
2,012023,082023,TechCorp,CloudDiv,Engineer,USA,107142.860000,75000.000000,USD,79380.000000,...,032021-122021,90000.000000,32142.860000,0.300000,Global Tax Engine v1.2,https://tax-engine.example.com/api/v1,NaN,NaN,NaN,NaN
0,032021,122021,TechCorp,CloudDiv,Engineer,USA,95000.000000,NaN,USD,90000.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [99]:
def aggregate_employment_summary(df):
    """
    Groups the processed dataframe by employer details and calculates 
    totals and annualized averages for both income sets.
    """
    df_copy = df.copy()
    
    # 1. Internal helper for grouping columns
    group_cols = ['Parent_employer_name', 'Subsidiary_employer_name', 'Designation', 'Country']
    
    # 2. Ensure we have tenure calculated for every row
    def get_row_tenure(row):
        return calculate_tenure_months(row['Start_employment'], row['End_employment'])
    
    df_copy['row_tenure_months'] = df_copy.apply(get_row_tenure, axis=1)

    # 3. Create helper columns for weighted averages (Income * Tenure)
    # This helps us handle partial years correctly in the final aggregation
    income_sets = {
        'Primary': ['Primary_income_pretax', 'Primary_income_posttax', 'Primary_income_currency'],
        'Client': ['Client_declared_pretax', 'Client_declared_posttax', 'Client_declared_currency']
    }

    # 4. Aggregation Logic
    # We use a custom function to handle the "ignore NaN rows for tenure" rule
    def group_logic(group):
        res = {}
        for prefix, cols in income_sets.items():
            pre_col, post_col, curr_col = cols
            
            # TOTALS (Sum of all non-NaN values)
            res[f'Total_{prefix}_Pretax'] = group[pre_col].sum()
            res[f'Total_{prefix}_Posttax'] = group[post_col].sum()
            res[f'Currency_{prefix}'] = group[curr_col].iloc[0] if group[curr_col].notnull().any() else np.nan
            
            # ANNUALIZED AVERAGES
            # Average = (Sum of Income / Total Months with Income) * 12
            
            # Pre-tax Average
            pre_valid_mask = group[pre_col].notnull()
            pre_tenure_sum = group.loc[pre_valid_mask, 'row_tenure_months'].sum()
            if pre_tenure_sum > 0:
                res[f'Annualized_Avg_{prefix}_Pretax'] = (group[pre_col].sum() / pre_tenure_sum) * 12
            else:
                res[f'Annualized_Avg_{prefix}_Pretax'] = np.nan
                
            # Post-tax Average
            post_valid_mask = group[post_col].notnull()
            post_tenure_sum = group.loc[post_valid_mask, 'row_tenure_months'].sum()
            if post_tenure_sum > 0:
                res[f'Annualized_Avg_{prefix}_Posttax'] = (group[post_col].sum() / post_tenure_sum) * 12
            else:
                res[f'Annualized_Avg_{prefix}_Posttax'] = np.nan
                
        return pd.Series(res)

    # Perform the GroupBy
    aggregated_df = df_copy.groupby(group_cols, observed=True).apply(group_logic).reset_index()
    
    return aggregated_df

# Execute Final Step
df_summary = aggregate_employment_summary(df_final)

C:\Users\User\AppData\Local\Temp\ipykernel_13828\865931323.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aggregated_df = df_copy.groupby(group_cols, observed=True).apply(group_logic).reset_index()


In [100]:
df_summary

,Parent_employer_name,Subsidiary_employer_name,Designation,Country,Total_Primary_Pretax,Total_Primary_Posttax,Currency_Primary,Annualized_Avg_Primary_Pretax,Annualized_Avg_Primary_Posttax,Total_Client_Pretax,Total_Client_Posttax,Currency_Client,Annualized_Avg_Client_Pretax,Annualized_Avg_Client_Posttax
0,FinanceHub,BankNodes,Analyst,UK,180555.550000,126388.888889,GBP,74712.641379,52298.850575,152142.857143,0.000000,GBP,62955.665025,NaN
1,RetailCorp,StoreFront,Manager,India,3431292.517007,0.000000,INR,1143764.172336,NaN,3676384.850000,2573469.387755,INR,1225461.616667,857823.129252
2,TechCorp,CloudDiv,Engineer,USA,305000.000000,147000.000000,USD,122000.000000,88200.000000,266522.860000,68000.000000,USD,106609.144000,68000.000000
